# HYDRA experiments

Choose an experiment, preview the runs it will start, then launch them with Submitit on
**HYDRA at TU Berlin** (or locally). For a small example, start with
[introductory.ipynb](introductory.ipynb).

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

import yaml
from dotenv import load_dotenv

from tfm_finetuning.experiments.execution.bundle import write_bundle
from tfm_finetuning.experiments.execution.planning import build_plan
from tfm_finetuning.experiments.execution.submission import submit_plan
from tfm_finetuning.experiments.execution.launch import show_plan
from tfm_finetuning.utils.config import read_yaml_mapping
from tfm_finetuning.utils.paths import get_paths, get_repo_root

root = get_repo_root()
load_dotenv(root / ".env", override=False)
results = (root / get_paths().results_root).resolve()

## 1. Choose an experiment and preview its runs

Set `EXPERIMENT_CONFIG` to an experiment YAML from [configs/experiments](../configs/experiments).
where the datasets, models, preprocessing options and training settings are defined. `DATASETS`, `MODELS` and
`PREPROCESSING` can be set to further filter the choices defined in the file and `None` keeps all.

`SCOPE = "smoke"` runs a preliminary smoke test of the selected training configuration for two epochs and fewer splits and seeds as an execution check. Use `BACKEND = "slurm"` for launching the experiments on SLURM (use HYDRA default settings in
[hydra_cluster.yaml](../configs/slurm/hydra_cluster.yaml) or copy and edit it for another cluster)
or `"local"` to run on this machine. Previewing launches nothing, and the last lines show the first run's complete settings.

In [ ]:
EXPERIMENT_CONFIG = root / "configs/experiments/distribution_shift_label_noise.yaml"

# Optional filters of the choices in EXPERIMENT_CONFIG. None keeps its selection.
DATASETS = ["jm1"]
MODELS = ["tabpfnv25"]
PREPROCESSING = None
SCOPE = "full"
BACKEND = "slurm"  # Use "local" to run the jobs locally, or "slurm" to submit them to a SLURM cluster.
SLURM_CONFIG = root / "configs/slurm/hydra_cluster.yaml"  # SLURM settings. Adapt for other SLURM setups than HYDRA

if not EXPERIMENT_CONFIG.is_file():
    raise FileNotFoundError(EXPERIMENT_CONFIG)
experiment_name = EXPERIMENT_CONFIG.stem

if BACKEND not in {"local", "slurm"}:
    raise ValueError("BACKEND must be local or slurm")
cluster = read_yaml_mapping(SLURM_CONFIG) if BACKEND == "slurm" else None
launch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
plan = build_plan(
    root, experiment_name, cluster,
    results / "launches" / f"{experiment_name}-{BACKEND}-{launch_id}",
    datasets=DATASETS, models=MODELS, preprocessing=PREPROCESSING,
    scope=SCOPE, results_root=results, config_path=EXPERIMENT_CONFIG,
)
show_plan(plan)

print(f"Experiment configuration: {EXPERIMENT_CONFIG}")
print(yaml.safe_dump(plan["jobs"][0]["runs"][0]["config"], sort_keys=False))

## 2. Launch

Set `SUBMIT = True` and run the cell. SLURM returns after submission, while local execution
waits for each job. Rerunning the same plan is blocked, so preview again for a new launch.
Datasets are prepared automatically. Download the TabPFN weights first with
`uv run python -m tfm_finetuning.data.download.tabpfn_checkpoints`. Transfer experiments
need earlier runs to finish first, see [transfer prerequisites](../docs/transfer.md).

In [ ]:
SUBMIT = False  # Set True to execute the previewed plan.

if SUBMIT:
    write_bundle(plan)
    jobs = submit_plan(plan)
    submitted_plan = plan
    print(f"Launched {len(jobs)} jobs. Records: {plan['output']}/jobs.json")
else:
    print("Preview only. Set SUBMIT = True in this cell to launch the selected plan.")

## 3. Check jobs

Rerun the cell to refresh the job states and show the first job's logs (change the index to
inspect another run). Use `jobs[i].result()` to wait for a result and `jobs[i].cancel()` to
cancel. Results are saved under `results/<study>/<run_id>/` (or `RESULTS_ROOT`), launch
folders record plans, job IDs and logs, and completed runs are skipped on rerun.

In [ ]:
if "jobs" not in globals():
    print("No jobs submitted in this kernel yet.")
else:
    print(f"Launch: {submitted_plan['output']}")
    display([{"job_id": job.job_id, "state": job.state, "done": job.done()} for job in jobs])
    job = jobs[0]
    print(f"stdout: {job.paths.stdout}\nstderr: {job.paths.stderr}")
    for path, read in ((job.paths.stdout, job.stdout), (job.paths.stderr, job.stderr)):
        if path.exists():
            print(read())